# Mangrove Pulse: drone sampling energy study
Experiment MP-DRONE-ENERGY-01. This source copy has no stored outputs.

Run the cells in order, or choose Runtime > Run all. The final cell downloads the results ZIP.

When can water-supported contact sampling use less mission energy than hovering contact sampling after accounting for added equipment mass and landing/takeoff energy?

This is an analytical sensitivity study, not a flight simulation, working drone, ecological validation or comparison with proprietary UAE drone specifications. All mission/design values below are hypothetical. Equal observation quality is assumed, not demonstrated. Energy-budget eligibility does not establish safety or operational feasibility.

The completed satellite PoC remains separate: Sentinel-2/GMW reference agreement at Khor Kalba, 20 and 30 November 2018, threshold 0.49 and frozen north/south split with 200 m gap. It is not a health diagnosis or a planting prescription. C001 is a historical 11-cell disagreement, not a verified landing/planting site. Present-day sampling cannot directly validate 2018 cover.

The full concept includes flight, surface-water and mud mobility, sensing and conditional planting. This notebook isolates one reduced sampling variant. It does not validate the full concept, survival improvement, novelty, field access, carbon gain or superiority over the UAE programme.

Each run creates a new output folder rather than loading earlier local diagnostic results.

## Evidence and assumptions

The original Deep Research report is preserved outside this notebook. Its missing references, misdescription of MEDUSA and inconsistent area/contact-pressure/energy arithmetic must not be adopted as evidence.

Verified starting sources:
- [Farinha et al. (2022), MEDUSA](https://www.frontiersin.org/journals/environmental-science/articles/10.3389/fenvs.2022.1023269/full), DOI 10.3389/fenvs.2022.1023269: sections 2.1 and 4 document water landing/sampling and a mass-versus-dwell trade-off, with calm-water limitations. The published 18% flight-time increase without landing gear is context, **not our calibration**.
- [Qin et al. (2023), multimode aerial–aquatic robot](https://advanced.onlinelibrary.wiley.com/doi/full/10.1002/aisy.202300193), DOI 10.1002/aisy.202300193: section 2 and Figure 3 document lake surface-to-air transitions. This does not establish mangrove mud/saltwater performance.
- [Johnson (2004), NASA rotor theory](https://rotorcraft.arc.nasa.gov/Publications/files/Johnson_AHS-SF2004.pdf), pp. 2–3: ideal hover velocity and power relations. The lumped loss multiplier used here is **not a published specification or measured efficiency**.
- [EAD (20 January 2022)](https://www.ead.gov.ae/en/Media-Centre/News/Phase-2-Mangrove-Rehabilitation-Project): assessment, mapping, tailored dispersal and monitoring already form part of the prior UAE work. This model does not compare with its undisclosed component specifications.

The scientific papers were checked against publisher full text and the NASA source against its institutional PDF. This notebook records those checks; it does not retrieve literature at runtime or independently validate the team's original satellite results.

## Model and fair-comparison limits

Two platforms perform the same hypothetical observation task, at the same sites and with equal per-site dwell time:
- Hovering uses the shared sampling payload, flight battery and base aircraft mass.
- Supported operation uses the same base aircraft/payload/battery plus an entire additional water-support kit.

Both have equal disk area and the same uncalibrated loss multiplier. The complete round-trip/inter-site travel budget uses approximate hover-equivalent power, not a forward-flight/range prediction. The supported variant adds landing/takeoff time at each site. Its supported power includes electronics and possible station-keeping.

With mass m, gravity g, air density rho and effective rotor area A:
```
P_ideal = (m*g)^(3/2) / sqrt(2*rho*A)
P_air = k*P_ideal + P_common
E_hover = P0*(T_travel + N*t_dwell) / 3600
E_support = P1*(T_travel + N*t_transition) / 3600
            + P_surface*N*t_dwell/3600 + E_extra
t_break_even = [(P1-P0)*T_travel + P1*N*t_transition + 3600*E_extra]
               / [N*(P0-P_surface)]
```
Power is W; durations are seconds; energy is Wh. Savings require dwell strictly above the threshold when P0 > P_surface. If P_surface >= P0, this model has no finite beneficial dwell threshold.

Nominal battery energy × usable fraction sets the energy budget. The default withholds 20% instead of adding a second reserve surcharge. Battery capacity and mass are hypothetical paired inputs, not a closed battery design or peak-current check.

No wave/current model, wet-rotor losses, corrosion, station-keeping validation, drift, failed transitions, sediment contact, cleaning/retrieval logistics or sensor-quality measurements are represented. Extra-overhead sensitivity exposes one missing-energy risk; it is not a reliability model. A lower-energy result cannot establish equal accepted observations or overall superiority.

The fixed sensitivity grid tests multiple masses, travel/dwell durations, site counts, surface loads, loss factors and transition times. Its counts are not probabilities or confidence estimates.

## 1. Optional requested-case inputs

The default is a hypothetical example, not our drone specification. Leave it unchanged for the initial reproducibility run. Any edits are recorded in the downloadable assumptions and results; the fixed grid is unchanged.

In [ ]:
# Optional changes affect only the additional requested case.
# The predefined illustrative cases and fixed sensitivity grid stay unchanged.
# Every value below is hypothetical, not a component or mission specification.
REQUESTED_CASE_OVERRIDES = {
    "baseline_mass_kg": 8.0,          # Includes shared payload and battery.
    "added_support_mass_kg": 2.0,     # Complete added kit, not floats alone.
    "disk_area_m2": 0.8,
    "loss_multiplier": 1.6,          # Lumped scenario factor, not efficiency.
    "common_air_load_w": 25.0,
    "supported_total_power_w": 40.0, # Includes all sampling/holding loads.
    "travel_time_s": 300.0,          # Includes return and inter-site travel.
    "sites": 3,
    "dwell_per_site_s": 150.0,       # Same observation task for both variants.
    "transition_per_site_s": 30.0,   # Added landing + takeoff at each site.
    "additional_support_overhead_wh": 0.0,
    "battery_nominal_wh": 240.0,
    "battery_usable_fraction": 0.8,
    "air_density_kg_m3": 1.225,       # Reference constant, not site weather.
    "gravity_m_s2": 9.81,
}


## 2. Load the embedded model and tests

The notebook contains its source and creates a unique output folder for each full run.

In [ ]:
import hashlib
import importlib.util
import io
import json
import platform
import sys
import tempfile
import unittest
import zipfile
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import Markdown, display

MODEL_SOURCE = "\"\"\"Analytical sensitivity model, not a flight or ecological simulation.\n\nAll mission/design inputs are hypothetical. No satellite data or external packages\nare used. Run with Python 3.10+; see the Colab bundle's EVIDENCE_AND_MODEL.md.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import asdict, dataclass, replace\nimport hashlib\nfrom itertools import product\nimport json\nimport math\nfrom pathlib import Path\n\n\n@dataclass(frozen=True)\nclass Scenario:\n    baseline_mass_kg: float = 8.0  # Includes battery and shared sampling payload.\n    added_support_mass_kg: float = 2.0  # Complete additional kit, not just floats.\n    disk_area_m2: float = 0.8  # Equal effective non-overlapping area for both.\n    loss_multiplier: float = 1.6  # Hypothetical lumped factor, NOT an efficiency.\n    common_air_load_w: float = 25.0\n    supported_total_power_w: float = 40.0  # Includes the shared sampling loads.\n    travel_time_s: float = 300.0  # Complete round trip/inter-site flight budget.\n    sites: int = 3\n    dwell_per_site_s: float = 150.0  # Equal accepted-observation time assumption.\n    transition_per_site_s: float = 30.0  # Additional landing + takeoff air time.\n    additional_support_overhead_wh: float = 0.0  # Extra energy sensitivity term.\n    battery_nominal_wh: float = 240.0  # Included in baseline mass; NOT mass-sized.\n    battery_usable_fraction: float = 0.8  # Withhold 20%; do not also add 20% energy.\n    air_density_kg_m3: float = 1.225\n    gravity_m_s2: float = 9.81\n\n\ndef validate(scenario: Scenario) -> None:\n    \"\"\"Reject invalid/nonfinite inputs instead of emitting misleading results.\"\"\"\n    for name, value in asdict(scenario).items():\n        if isinstance(value, bool) or not math.isfinite(value):\n            raise ValueError(f\"{name} must be finite and numeric\")\n        if value < 0:\n            raise ValueError(f\"{name} cannot be negative\")\n    for name in (\"baseline_mass_kg\", \"disk_area_m2\", \"battery_nominal_wh\",\n                 \"air_density_kg_m3\", \"gravity_m_s2\"):\n        if getattr(scenario, name) <= 0:\n            raise ValueError(f\"{name} must be positive\")\n    if not isinstance(scenario.sites, int) or scenario.sites <= 0:\n        raise ValueError(\"sites must be a positive integer\")\n    if scenario.loss_multiplier < 1:\n        raise ValueError(\"loss_multiplier must be at least 1\")\n    if not 0 < scenario.battery_usable_fraction <= 1:\n        raise ValueError(\"battery_usable_fraction must be in (0, 1]\")\n\n\ndef induced_power_w(mass_kg: float, disk_area_m2: float,\n                    air_density_kg_m3: float = 1.225,\n                    gravity_m_s2: float = 9.81) -> float:\n    \"\"\"Ideal, static, out-of-ground-effect induced power; omits real losses.\"\"\"\n    for value in (mass_kg, disk_area_m2, air_density_kg_m3, gravity_m_s2):\n        if not math.isfinite(value) or value <= 0:\n            raise ValueError(\"power-model inputs must be positive and finite\")\n    return (mass_kg * gravity_m_s2) ** 1.5 / math.sqrt(\n        2 * air_density_kg_m3 * disk_area_m2\n    )\n\n\ndef evaluate(scenario: Scenario) -> dict:\n    \"\"\"Compare equal dwell tasks and derive the per-site break-even duration.\"\"\"\n    validate(scenario)\n    s = scenario\n    p0_ideal = induced_power_w(s.baseline_mass_kg, s.disk_area_m2,\n                             s.air_density_kg_m3, s.gravity_m_s2)\n    p1_ideal = induced_power_w(s.baseline_mass_kg + s.added_support_mass_kg,\n                             s.disk_area_m2, s.air_density_kg_m3, s.gravity_m_s2)\n    p0 = s.loss_multiplier * p0_ideal + s.common_air_load_w\n    p1 = s.loss_multiplier * p1_ideal + s.common_air_load_w\n    total_dwell_s = s.sites * s.dwell_per_site_s\n    total_transition_s = s.sites * s.transition_per_site_s\n    hover_travel_wh = p0 * s.travel_time_s / 3600\n    hover_dwell_wh = p0 * total_dwell_s / 3600\n    support_travel_wh = p1 * s.travel_time_s / 3600\n    support_transition_wh = p1 * total_transition_s / 3600\n    support_dwell_wh = s.supported_total_power_w * total_dwell_s / 3600\n    hover_wh = hover_travel_wh + hover_dwell_wh\n    support_wh = (support_travel_wh + support_transition_wh + support_dwell_wh\n                  + s.additional_support_overhead_wh)\n    saving_wh = hover_wh - support_wh\n    numerator_ws = ((p1 - p0) * s.travel_time_s + p1 * total_transition_s\n                    + 3600 * s.additional_support_overhead_wh)\n    denominator_w = s.sites * (p0 - s.supported_total_power_w)\n    threshold_s = numerator_ws / denominator_w if denominator_w > 0 else None\n    usable_wh = s.battery_nominal_wh * s.battery_usable_fraction\n    # A budget pass is not a flight, transition, access, quality or safety pass.\n    return {\n        \"baseline_ideal_induced_power_w\": p0_ideal,\n        \"supported_ideal_induced_power_w\": p1_ideal,\n        \"baseline_air_power_w\": p0,\n        \"supported_air_power_w\": p1,\n        \"hover_segments_wh\": {\"round_trip_travel\": hover_travel_wh,\n                              \"sampling_hover\": hover_dwell_wh},\n        \"supported_segments_wh\": {\"round_trip_travel\": support_travel_wh,\n                                  \"land_takeoff\": support_transition_wh,\n                                  \"supported_sampling\": support_dwell_wh,\n                                  \"additional_overhead\": s.additional_support_overhead_wh},\n        \"hover_energy_wh\": hover_wh,\n        \"supported_energy_wh\": support_wh,\n        \"saving_wh\": saving_wh,\n        \"saving_percent_of_hover\": 100 * saving_wh / hover_wh if hover_wh else None,\n        \"break_even_dwell_per_site_s\": threshold_s,\n        \"supported_energy_lower\": saving_wh > 1e-9,\n        \"usable_battery_wh\": usable_wh,\n        \"hover_within_energy_budget\": hover_wh <= usable_wh + 1e-9,\n        \"supported_within_energy_budget\": support_wh <= usable_wh + 1e-9,\n        \"hover_accounted_duration_s\": s.travel_time_s + total_dwell_s,\n        \"supported_accounted_duration_s\": s.travel_time_s + total_dwell_s + total_transition_s,\n        \"duration_note\": \"Subtotal; additional-overhead duration is unspecified if nonzero\",\n        \"operational_feasibility_established\": False,\n        \"observation_equivalence_verified\": False,\n    }\n\n\ndef original_report_arithmetic() -> dict:\n    \"\"\"Reproduce and correct the report's explicit arithmetic, not its design.\"\"\"\n    area = 4 * math.pi * 0.4 ** 2\n    segments_wh = [1600 * 100 / 3600, 1800 * 300 / 3600,\n                   200 * 100 / 3600, 100 * 60 / 3600, 1800 * 300 / 3600]\n    subtotal = sum(segments_wh)\n    return {\n        \"report_sha256\": \"d8da47734a872a7408b741119fa576cc167011ef65d84366bc244fbba47fe224\",\n        \"four_rotor_disk_area_m2\": area,\n        \"hover_table\": [{\"total_mass_kg\": m, \"ideal_induced_power_w\":\n                         induced_power_w(m, area)} for m in (23, 27, 31, 35, 43)],\n        \"area_of_50_by_100_m_km2\": 50 * 100 / 1_000_000,\n        \"two_20_by_5_cm_contact_patches_m2\": 2 * 0.2 * 0.05,\n        \"contact_pressure_30_kg_kpa\": 30 * 9.81 / (2 * 0.2 * 0.05) / 1000,\n        \"segment_energy_subtotal_wh\": subtotal,\n        \"segment_energy_with_10_percent_addition_wh\": subtotal * 1.1,\n        \"fraction_of_750_wh_after_addition\": subtotal * 1.1 / 750,\n        \"flight_segment_duration_min\": (100 + 300 + 300) / 60,\n        \"20_percent_added_mass_ideal_power_ratio\": 1.2 ** 1.5,\n        \"23_to_27_kg_added_mass_percent\": 100 * (27 / 23 - 1),\n        \"23_to_27_kg_ideal_power_ratio\": (27 / 23) ** 1.5,\n    }\n\n\ndef build_results() -> dict:\n    base = Scenario()\n    cases = {\n        \"short_dwell\": replace(base, dwell_per_site_s=30),\n        \"illustrative_main\": base,\n        \"long_dwell\": replace(base, dwell_per_site_s=300),\n        \"long_route_energy_only_win\": replace(base, travel_time_s=1200,\n                                               dwell_per_site_s=300),\n        \"extra_80_wh_overhead\": replace(base, additional_support_overhead_wh=80),\n    }\n    axes = {\n        \"added_support_mass_kg\": [1.0, 2.0, 4.0],\n        \"travel_time_s\": [300.0, 600.0, 1200.0],\n        \"sites\": [1, 3, 5],\n        \"dwell_per_site_s\": [30.0, 60.0, 150.0, 300.0],\n        \"supported_total_power_w\": [40.0, 150.0],\n        \"loss_multiplier\": [1.3, 1.8],\n        \"transition_per_site_s\": [15.0, 60.0],\n    }\n    rows = []\n    for values in product(*axes.values()):\n        scenario = replace(base, **dict(zip(axes, values)))\n        rows.append({\"inputs\": asdict(scenario), \"result\": evaluate(scenario)})\n    return {\n        \"model_version\": \"1.0\",\n        \"evidence_status\": \"Analytical scenario estimates, not measured drone performance\",\n        \"input_provenance\": {\n            \"mission_and_design_values\": \"Hypothetical sensitivity inputs; no component specifications\",\n            \"air_density_and_gravity\": \"Chosen reference constants, not measured Khor Kalba conditions\",\n            \"power_equation\": \"Ideal actuator-disk momentum theory; Johnson (2004), pp. 2–3\",\n            \"power_source_url\": \"https://rotorcraft.arc.nasa.gov/Publications/files/Johnson_AHS-SF2004.pdf\",\n            \"loss_multiplier\": \"Uncalibrated lumped scenario factor, not a published motor efficiency\",\n            \"observation_equivalence\": \"Assumed; no sensor quality, depth or stability measurements\",\n            \"published_18_percent_MEDUSA_result\": \"Context only; NOT a model calibration or input\",\n        },\n        \"original_report_arithmetic_checks\": original_report_arithmetic(),\n        \"illustrative_cases\": {name: {\"inputs\": asdict(s), \"result\": evaluate(s)}\n                               for name, s in cases.items()},\n        \"sweep_axes\": axes,\n        \"sweep_summary\": {\n            \"scenarios\": len(rows),\n            \"energy_lower\": sum(r[\"result\"][\"supported_energy_lower\"] for r in rows),\n            \"energy_lower_and_supported_budget_pass\": sum(\n                r[\"result\"][\"supported_energy_lower\"] and\n                r[\"result\"][\"supported_within_energy_budget\"] for r in rows),\n            \"interpretation\": \"Grid counts are not probabilities or evidence of real mission success\",\n        },\n        \"sweep\": rows,\n    }\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--output\", type=Path,\n                        default=Path(__file__).with_name(\"sampling_energy_results.json\"))\n    args = parser.parse_args()\n    results = build_results()\n    results[\"model_source_sha256\"] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    args.output.write_text(json.dumps(results, indent=2, allow_nan=False) + \"\\n\", encoding=\"utf-8\")\n    print(json.dumps({\"output\": str(args.output.resolve()),\n                      \"sweep_summary\": results[\"sweep_summary\"],\n                      \"illustrative_cases\": {name: case[\"result\"] for name, case in\n                                             results[\"illustrative_cases\"].items()}}, indent=2))\n\n\nif __name__ == \"__main__\":\n    main()\n"
TEST_SOURCE = "\"\"\"Unit, conservation, boundary and sensitivity checks for the analytical model.\"\"\"\n\nfrom dataclasses import replace\nimport math\nimport unittest\n\nfrom sampling_energy_model import (Scenario, build_results, evaluate,\n                                   induced_power_w, original_report_arithmetic)\n\n\nclass SamplingEnergyTests(unittest.TestCase):\n    def test_original_hover_table(self):\n        area = 4 * math.pi * 0.4 ** 2\n        self.assertAlmostEqual(induced_power_w(23, area), 1527.0290516837522)\n        self.assertAlmostEqual(induced_power_w(43, area), 3903.5363591526134)\n\n    def test_mass_and_area_scaling(self):\n        self.assertAlmostEqual(induced_power_w(9.6, 0.8) / induced_power_w(8, 0.8),\n                               1.2 ** 1.5)\n        self.assertAlmostEqual(induced_power_w(8, 3.2), induced_power_w(8, 0.8) / 2)\n\n    def test_independent_hand_case_and_unit_conversion(self):\n        s = replace(Scenario(), baseline_mass_kg=1, added_support_mass_kg=0,\n                    disk_area_m2=0.5, air_density_kg_m3=1, gravity_m_s2=1,\n                    loss_multiplier=1, common_air_load_w=0, supported_total_power_w=0,\n                    travel_time_s=3600, sites=1, dwell_per_site_s=3600,\n                    transition_per_site_s=0)\n        result = evaluate(s)  # Ideal power = 1 W, independent of production equation.\n        self.assertAlmostEqual(result[\"hover_energy_wh\"], 2)\n        self.assertAlmostEqual(result[\"supported_energy_wh\"], 1)\n\n    def test_segment_conservation(self):\n        result = evaluate(Scenario())\n        self.assertAlmostEqual(sum(result[\"hover_segments_wh\"].values()),\n                               result[\"hover_energy_wh\"])\n        self.assertAlmostEqual(sum(result[\"supported_segments_wh\"].values()),\n                               result[\"supported_energy_wh\"])\n        self.assertAlmostEqual(result[\"saving_wh\"], result[\"hover_energy_wh\"] -\n                               result[\"supported_energy_wh\"])\n\n    def test_break_even_and_both_sides(self):\n        base = Scenario()\n        threshold = evaluate(base)[\"break_even_dwell_per_site_s\"]\n        self.assertGreater(threshold, 0)\n        self.assertAlmostEqual(evaluate(replace(base, dwell_per_site_s=threshold))[\"saving_wh\"], 0)\n        self.assertLess(evaluate(replace(base, dwell_per_site_s=threshold / 2))[\"saving_wh\"], 0)\n        self.assertGreater(evaluate(replace(base, dwell_per_site_s=threshold * 2))[\"saving_wh\"], 0)\n\n    def test_identical_power_no_benefit(self):\n        base = replace(Scenario(), added_support_mass_kg=0, transition_per_site_s=0)\n        p = evaluate(base)[\"baseline_air_power_w\"]\n        result = evaluate(replace(base, supported_total_power_w=p))\n        self.assertIsNone(result[\"break_even_dwell_per_site_s\"])\n        self.assertAlmostEqual(result[\"saving_wh\"], 0)\n        self.assertFalse(result[\"supported_energy_lower\"])\n\n    def test_supported_power_above_hover_never_wins(self):\n        base = Scenario()\n        p = evaluate(base)[\"baseline_air_power_w\"]\n        result = evaluate(replace(base, supported_total_power_w=p + 1))\n        self.assertIsNone(result[\"break_even_dwell_per_site_s\"])\n        self.assertLess(result[\"saving_wh\"], 0)\n\n    def test_free_support_zero_penalty(self):\n        base = replace(Scenario(), added_support_mass_kg=0, transition_per_site_s=0)\n        self.assertEqual(evaluate(base)[\"break_even_dwell_per_site_s\"], 0)\n        self.assertTrue(evaluate(base)[\"supported_energy_lower\"])\n\n    def test_sensitivity_directions(self):\n        base = Scenario()\n        threshold = evaluate(base)[\"break_even_dwell_per_site_s\"]\n        for change in ({\"added_support_mass_kg\": 4}, {\"travel_time_s\": 600},\n                       {\"transition_per_site_s\": 60}, {\"supported_total_power_w\": 150},\n                       {\"additional_support_overhead_wh\": 80}):\n            self.assertGreater(evaluate(replace(base, **change))[\"break_even_dwell_per_site_s\"],\n                               threshold)\n        self.assertLess(evaluate(replace(base, sites=5))[\"break_even_dwell_per_site_s\"], threshold)\n\n    def test_budget_gate_is_not_savings_gate(self):\n        result = evaluate(replace(Scenario(), travel_time_s=1200, dwell_per_site_s=300))\n        self.assertTrue(result[\"supported_energy_lower\"])\n        self.assertFalse(result[\"supported_within_energy_budget\"])\n        self.assertFalse(result[\"operational_feasibility_established\"])\n\n    def test_reserve_withheld_not_double_counted(self):\n        result = evaluate(Scenario())\n        self.assertAlmostEqual(result[\"usable_battery_wh\"], 192)\n        s = replace(Scenario(), battery_usable_fraction=1)\n        self.assertEqual(evaluate(s)[\"hover_energy_wh\"], result[\"hover_energy_wh\"])\n\n    def test_time_accounts_for_transitions(self):\n        result = evaluate(Scenario())\n        self.assertEqual(result[\"supported_accounted_duration_s\"] -\n                         result[\"hover_accounted_duration_s\"], 90)\n\n    def test_report_arithmetic(self):\n        result = original_report_arithmetic()\n        self.assertAlmostEqual(result[\"area_of_50_by_100_m_km2\"], 0.005)\n        self.assertAlmostEqual(result[\"contact_pressure_30_kg_kpa\"], 14.715)\n        self.assertAlmostEqual(result[\"segment_energy_with_10_percent_addition_wh\"], 386.8333333333333)\n        self.assertAlmostEqual(result[\"flight_segment_duration_min\"], 11 + 2 / 3)\n\n    def test_invalid_inputs(self):\n        for changes in ({\"sites\": 0}, {\"sites\": 1.5}, {\"sites\": True},\n                        {\"added_support_mass_kg\": -1}, {\"disk_area_m2\": 0},\n                        {\"battery_usable_fraction\": 0}, {\"battery_usable_fraction\": 1.1},\n                        {\"loss_multiplier\": 0.9}, {\"travel_time_s\": math.nan},\n                        {\"dwell_per_site_s\": math.inf}, {\"supported_total_power_w\": -1}):\n            with self.subTest(changes=changes), self.assertRaises(ValueError):\n                evaluate(replace(Scenario(), **changes))\n        with self.assertRaises(ValueError):\n            induced_power_w(0, 1)\n\n    def test_sweep_and_determinism(self):\n        results = build_results()\n        self.assertEqual(results, build_results())\n        self.assertEqual(results[\"sweep_summary\"][\"scenarios\"], 864)\n        statuses = {row[\"result\"][\"supported_energy_lower\"] for row in results[\"sweep\"]}\n        self.assertEqual(statuses, {False, True})\n        for row in results[\"sweep\"]:\n            s, result = row[\"inputs\"], row[\"result\"]\n            threshold = result[\"break_even_dwell_per_site_s\"]\n            if threshold is not None and abs(s[\"dwell_per_site_s\"] - threshold) > 1e-6:\n                self.assertEqual(result[\"supported_energy_lower\"], s[\"dwell_per_site_s\"] > threshold)\n\n\nif __name__ == \"__main__\":\n    unittest.main()\n"
SOURCE_NOTES = "## Evidence and assumptions\n\nThe original Deep Research report is preserved outside this notebook. Its missing references, misdescription of MEDUSA and inconsistent area/contact-pressure/energy arithmetic must not be adopted as evidence.\n\nVerified starting sources:\n- [Farinha et al. (2022), MEDUSA](https://www.frontiersin.org/journals/environmental-science/articles/10.3389/fenvs.2022.1023269/full), DOI 10.3389/fenvs.2022.1023269: sections 2.1 and 4 document water landing/sampling and a mass-versus-dwell trade-off, with calm-water limitations. The published 18% flight-time increase without landing gear is context, **not our calibration**.\n- [Qin et al. (2023), multimode aerial–aquatic robot](https://advanced.onlinelibrary.wiley.com/doi/full/10.1002/aisy.202300193), DOI 10.1002/aisy.202300193: section 2 and Figure 3 document lake surface-to-air transitions. This does not establish mangrove mud/saltwater performance.\n- [Johnson (2004), NASA rotor theory](https://rotorcraft.arc.nasa.gov/Publications/files/Johnson_AHS-SF2004.pdf), pp. 2–3: ideal hover velocity and power relations. The lumped loss multiplier used here is **not a published specification or measured efficiency**.\n- [EAD (20 January 2022)](https://www.ead.gov.ae/en/Media-Centre/News/Phase-2-Mangrove-Rehabilitation-Project): assessment, mapping, tailored dispersal and monitoring already form part of the prior UAE work. This model does not compare with its undisclosed component specifications.\n\nThe scientific papers are verified via publisher full text; the NASA source via its institutional PDF. This notebook records those checks, but does not retrieve literature at runtime or independently validate the team’s original satellite results.\n\n## Model and fair-comparison limits\n\nTwo platforms perform the same hypothetical observation task, at the same sites and with equal per-site dwell time:\n- **Hovering:** shared sampling payload, flight battery and base aircraft mass.\n- **Supported:** the same base aircraft/payload/battery plus an entire additional water-support kit.\n\nBoth have equal disk area and the same uncalibrated loss multiplier. The complete round-trip/inter-site travel budget is approximated at hover-equivalent power; this is **not a forward-flight/range prediction**. The supported variant adds landing/takeoff time at each site. Its supported power includes electronics and possible station-keeping.\n\nWith mass m, gravity g, air density rho and effective rotor area A:\n```\nP_ideal = (m*g)^(3/2) / sqrt(2*rho*A)\nP_air = k*P_ideal + P_common\nE_hover = P0*(T_travel + N*t_dwell) / 3600\nE_support = P1*(T_travel + N*t_transition) / 3600\n            + P_surface*N*t_dwell/3600 + E_extra\nt_break_even = [(P1-P0)*T_travel + P1*N*t_transition + 3600*E_extra]\n               / [N*(P0-P_surface)]\n```\nPower is W; durations are seconds; energy is Wh. Savings require dwell **strictly above** the threshold when P0 > P_surface. If P_surface >= P0, this model has no finite beneficial dwell threshold.\n\nNominal battery energy × usable fraction sets the energy budget. The default withholds 20% instead of adding a second reserve surcharge. Battery capacity and mass are hypothetical paired inputs, not a closed battery design or peak-current check.\n\nNo wave/current model, wet-rotor losses, corrosion, station-keeping validation, drift, failed transitions, sediment contact, cleaning/retrieval logistics or sensor-quality measurements are represented. Extra-overhead sensitivity exposes one missing-energy risk; it is not a reliability model. A lower-energy result cannot establish equal accepted observations or overall superiority.\n\nThe fixed sensitivity grid tests multiple masses, travel/dwell durations, site counts, surface loads, loss factors and transition times. Its counts are **not probabilities or confidence estimates**."

# A unique, task-specific folder prevents stale output mixing. No deletion.
RUN_DIR = Path(tempfile.mkdtemp(prefix="mangrove_pulse_drone_"))
MODEL_PATH = RUN_DIR / "sampling_energy_model.py"
TEST_PATH = RUN_DIR / "test_sampling_energy_model.py"
MODEL_PATH.write_text(MODEL_SOURCE, encoding="utf-8")
TEST_PATH.write_text(TEST_SOURCE, encoding="utf-8")
(RUN_DIR / "EVIDENCE_AND_MODEL.md").write_text(SOURCE_NOTES, encoding="utf-8")

RUN_STATUS = {
    "experiment": "MP-DRONE-ENERGY-01",
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "status": "prepared",
    "execution_environment": "google_colab" if "google.colab" in sys.modules else "python_notebook",
    "python_version": sys.version,
    "platform": platform.platform(),
    "model_source_sha256": hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest(),
    "test_source_sha256": hashlib.sha256(TEST_PATH.read_bytes()).hexdigest(),
    "results_are": "Analytical estimates using hypothetical inputs, not drone measurements",
}

def save_status():
    (RUN_DIR / "run_status.json").write_text(
        json.dumps(RUN_STATUS, indent=2, allow_nan=False) + "\n", encoding="utf-8")

def load_module(name, path):
    specification = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(specification)
    sys.modules[name] = module
    specification.loader.exec_module(module)
    return module

# Scoped modules are loaded from the embedded source, not an external repository.
model = load_module("sampling_energy_model", MODEL_PATH)
save_status()
print("Fresh run folder:", RUN_DIR)
print("No geospatial data, external packages or source files are required.")


## 3. Verify the calculation engine

Checks cover arithmetic, units, segment conservation, equality/negative-benefit boundaries, monotonic sensitivity, battery gates and deterministic grid generation. Failed checks prevent calculations; diagnostic files can still be downloaded.

In [ ]:
test_module = load_module("test_sampling_energy_model", TEST_PATH)
test_stream = io.StringIO()
test_suite = unittest.defaultTestLoader.loadTestsFromModule(test_module)
test_result = unittest.TextTestRunner(stream=test_stream, verbosity=2).run(test_suite)
TESTS_PASSED = test_result.wasSuccessful()
TEST_SUMMARY = {
    "status": "passed" if TESTS_PASSED else "failed",
    "tests_run": test_result.testsRun,
    "failures": len(test_result.failures),
    "errors": len(test_result.errors),
    "skipped": len(test_result.skipped),
}
(RUN_DIR / "verification_log.txt").write_text(test_stream.getvalue(), encoding="utf-8")
(RUN_DIR / "verification_summary.json").write_text(
    json.dumps(TEST_SUMMARY, indent=2) + "\n", encoding="utf-8")
RUN_STATUS["verification"] = TEST_SUMMARY
RUN_STATUS["status"] = "tests_passed" if TESTS_PASSED else "tests_failed"
save_status()
print(test_stream.getvalue())
if not TESTS_PASSED:
    print("Calculations will be skipped. The final cell still packages diagnostics.")


## 4. Execute the illustrative cases and fixed sensitivity grid

All results are generated in this runtime. Missing physical measurements remain assumptions.

In [ ]:
RESULTS = None
REQUESTED_CASE = None
if TESTS_PASSED:
    try:
        # All calculation results in this package originate in this runtime.
        RESULTS = model.build_results()
        RESULTS["model_source_sha256"] = RUN_STATUS["model_source_sha256"]
        REQUESTED_CASE = model.Scenario(**REQUESTED_CASE_OVERRIDES)
        RESULTS["notebook_requested_case"] = {
            "inputs": asdict(REQUESTED_CASE),
            "result": model.evaluate(REQUESTED_CASE),
        }
        (RUN_DIR / "sampling_energy_results.json").write_text(
            json.dumps(RESULTS, indent=2, allow_nan=False) + "\n", encoding="utf-8")
        inputs_record = {
            "requested_case": asdict(REQUESTED_CASE),
            "default_case": asdict(model.Scenario()),
            "fixed_sweep_axes": RESULTS["sweep_axes"],
            "classification": "Hypothetical sensitivity inputs, not measured specifications",
        }
        (RUN_DIR / "assumptions.json").write_text(
            json.dumps(inputs_record, indent=2, allow_nan=False) + "\n", encoding="utf-8")
        RUN_STATUS["status"] = "calculations_completed"
        RUN_STATUS["sweep_summary"] = RESULTS["sweep_summary"]
        print(json.dumps(RESULTS["sweep_summary"], indent=2))
    except Exception as error:
        import traceback
        RESULTS = None
        RUN_STATUS["status"] = "calculation_failed"
        RUN_STATUS["error"] = str(error)
        (RUN_DIR / "calculation_error.txt").write_text(traceback.format_exc(), encoding="utf-8")
        print("Calculation failed; the final cell will download diagnostics:", error)
    save_status()
else:
    print("Skipped calculations because verification failed.")


## 5. Review the generated summary

The battery gate and energy-saving gate are separate. A lower-energy case may still exceed its battery budget.

In [ ]:
if RESULTS is not None:
    lines = [
        "# Mangrove Pulse: Colab analytical results",
        "",
        "**Status: hypothetical scenario estimates, not validated drone performance.**",
        "",
        "## Illustrative comparisons",
        "",
        "| Case | Hover Wh | Supported Wh | Saving Wh | Threshold s/site | Energy budget: hover / supported |",
        "|---|---:|---:|---:|---:|---|",
    ]
    all_cases = dict(RESULTS["illustrative_cases"])
    all_cases["notebook_requested_case"] = RESULTS["notebook_requested_case"]
    for name, case in all_cases.items():
        r = case["result"]
        threshold = r["break_even_dwell_per_site_s"]
        threshold_text = "none" if threshold is None else f"{threshold:.1f}"
        budget = (("pass" if r["hover_within_energy_budget"] else "fail") + " / " +
                  ("pass" if r["supported_within_energy_budget"] else "fail"))
        lines.append(
            f"| {name} | {r['hover_energy_wh']:.1f} | {r['supported_energy_wh']:.1f} | "
            f"{r['saving_wh']:.1f} | {threshold_text} | {budget} |"
        )
    lines += [
        "",
        "Positive saving means supported sampling uses less energy for the assumed equal task.",
        "A budget pass is not evidence of flight, water transitions, safe access or sensor quality.",
        "",
        "## Requested-case assumptions",
        "",
        "```json",
        json.dumps(asdict(REQUESTED_CASE), indent=2),
        "```",
        "",
        "## Fixed sensitivity grid",
        "",
        "```json",
        json.dumps(RESULTS["sweep_summary"], indent=2),
        "```",
        "",
        "Grid counts describe these arbitrary combinations, not real-world success probabilities.",
        "",
        "## Original report arithmetic checks",
        "",
        "```json",
        json.dumps(RESULTS["original_report_arithmetic_checks"], indent=2),
        "```",
        "",
        "The original report is not needed or uploaded. These checks reproduce its explicitly stated inputs.",
        "The stored report SHA256 identifies the reviewed version, not a runtime verification of that absent file.",
        "",
        "## Interpretation",
        "",
        "KEEP the auditable satellite-to-inspection question. CHANGE broad drone-superiority claims",
        "to this limited mass/transition/dwell trade-off. TEST real input ranges, supported-state power,",
        "transition energy and observation equivalence. DEFER mud mobility, planting, survival,",
        "carbon benefit and fleet claims; none is established by this model.",
        "",
        "This is not a comparison against the UAE programme's actual drone specifications,",
        "nor against a fully costed boat/manual workflow. No such superiority ranking is supported.",
        "",
        "See EVIDENCE_AND_MODEL.md for equations, source links and omitted effects.",
    ]
    summary_text = "\n".join(lines) + "\n"
    (RUN_DIR / "RESULTS_SUMMARY.md").write_text(summary_text, encoding="utf-8")
    display(Markdown(summary_text))
else:
    print("No successful model results. The downloadable package will contain diagnostics only.")


## 6. Package and download

The ZIP includes results, assumptions, Markdown evidence notes, calculation code, test logs, runtime details and file hashes. If anything failed, inspect the diagnostic files rather than treating the run as a successful study.

In [ ]:
# Package only this fresh task-specific run folder; never scan Drive or accounts.
RUN_STATUS["finished_utc"] = datetime.now(timezone.utc).isoformat()
if RESULTS is not None:
    RUN_STATUS["status"] = "completed_with_tests_passed"
save_status()

readme = """# Mangrove Pulse drone-energy result package
These files are analytical outputs, not measured robot or ecological performance.
Start with RESULTS_SUMMARY.md (if calculations succeeded) and run_status.json.
assumptions.json records the actual requested case and fixed sensitivity axes.
sampling_energy_results.json contains detailed illustrative and grid calculations.
EVIDENCE_AND_MODEL.md records equations, source context and limitations.
verification_summary.json and verification_log.txt record the checks performed.
The Python source and test module are included for audit and reproduction.

Reproduce the fixed grid with Python 3.10+:
    python -m unittest discover -s . -p test_sampling_energy_model.py -v
    python sampling_energy_model.py --output reproduced_default_results.json

The command above reproduces predefined cases and the fixed grid. To reproduce
a changed notebook_requested_case, load assumptions.json, construct Scenario
from its requested_case entry and call evaluate. The results JSON stores those inputs too.

SHA256SUMS.json covers the package files, excluding itself and the ZIP container.
The report hash in arithmetic checks is an identifier, not proof the original
report was uploaded or revalidated here. No original report, private history,
participant-only material, geospatial inputs, imagery pixels or PDFs are bundled.
"""
(RUN_DIR / "README.md").write_text(readme, encoding="utf-8")

package_files = sorted(p for p in RUN_DIR.iterdir() if p.is_file() and p.name != "SHA256SUMS.json")
manifest = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in package_files}
(RUN_DIR / "SHA256SUMS.json").write_text(
    json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
package_files.append(RUN_DIR / "SHA256SUMS.json")
BUNDLE_PATH = RUN_DIR.parent / (RUN_DIR.name + "_results.zip")
with zipfile.ZipFile(BUNDLE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(package_files):
        archive.write(p, arcname=p.name)

with zipfile.ZipFile(BUNDLE_PATH) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("ZIP integrity check failed")
    for name, expected in manifest.items():
        if hashlib.sha256(archive.read(name)).hexdigest() != expected:
            raise RuntimeError("Package hash mismatch: " + name)
print("Package checked:", BUNDLE_PATH.name)
print("Run status:", RUN_STATUS["status"])
print("Files included:", ", ".join(sorted(p.name for p in package_files)))


In [ ]:
from google.colab import files
files.download(str(BUNDLE_PATH))
# If the browser blocks the automatic download, rerun this cell.
# The ZIP also remains at BUNDLE_PATH in the Colab runtime until it is reset.
